In [1]:
import os
import pandas as pd
import json
import ast

from ollama import Client, generate
import requests, json, time
from typing import Dict, Any, List, Optional

In [2]:
df = pd.read_excel('NLP_annotated_sample_2000.xlsx')
df = df.drop(columns=(['Unnamed: 0.1', 'Unnamed: 0', 'atepc_sentiments', 'atepc_confidence', 'atepc_aspects', 'tmp']))
df.shape

(2000, 11)

In [3]:
df['len_manual'] = df['ABSA_manual'].apply(lambda x: len(ast.literal_eval(x).get('aspects', [])) if isinstance(x, str) else len(x.get('aspects', [])))
df['len_atepc'] = df['atepc_json'].apply(lambda x: len(ast.literal_eval(x).get('aspect', [])) if isinstance(x, str) else len(x.get('aspect', [])))

In [4]:
df.head(2)

,place_id,author,rating,date,text,place_name,index,date_parsed,ABSA_manual,atepc_json,flag,len_manual,len_atepc
0,ChIJh8eBJRShEmsRwG7p0IjG-kk,relpmat,1,6 years ago,I have always been a big fan of UFC and what t...,RIVAL GYM Castle Hill,11725,2019-10-12,"{""overall_sentiment"":""positive"",""aspects"":[{""a...",{'sentence': 'I have always been a big fan of ...,NaN,1,0
1,ChIJn8yEJI8U1moRVjM3zwJwCRg,michael k.,1,a year ago,Very unfriendly staff and service !,Australia Post - Noble Park Post Shop,15672,2024-10-10,"{""overall_sentiment"":""negative"",""aspects"":[{""a...",{'sentence': 'Very unfriendly staff and servic...,NaN,2,2


In [10]:
df[((abs(df.len_manual - df.len_atepc)==0) | (df.flag==1)) & (df.len_manual>0) & (df.len_manual<6)].to_excel('1000_annotated_sample_final.xlsx')

# Zero shot

## phi4:14b 

### Pipeline prompts
- aspect term extraction (ATE), which identifies the features or aspects being discussed (e.g., "food"), 
- opinion term extraction (OTE), which finds the words expressing an opinion (e.g., "delicious"), 
- aspect-level sentiment classification (ASC)

In [79]:
MODEL = "phi4:14b"
# MODEL = "llama3.2:1b"

In [67]:
# Schemas
ate_schema = """
{
  "aspects": [
    {
      "term": "string",            // ≤ 2 words, lemmatized if plural
      "category": "string"    // optional coarse tag (e.g., service, price)
    }
  ]
}"""
ote_schema = """
{
  "opinions": [
    {
      "aspect": string,           // aspect from ATE aspects[]
      "opinion": "string",         // ≤ 3 words
      "link_type": "explicit | implicit"
    }
  ]
}
"""
alsc_schema = """
{
  "sentiments": [
    {
      "aspect": string,           // aspect from ATE aspects[]
      "category": "string",       // category from ATE aspects[]
      "label": "positive|negative|neutral",
      "evidence_span": "string",   // short verbatim quote (≤ 6 words)
    }
  ],
  "overall_sentiment": "positive|negative|neutral"
}
"""

In [68]:
def ate(review_text:str, ate_schema:str):
    ate_prompt = f"""
    System:
    You are an expert annotator for Aspect Term Extraction (ATE). Extract explicit aspects as short noun phrases (≤ 2 words). Return valid JSON exactly matching the provided schema. Use character offsets for the original text.

    User:
    TEXT:
    {review_text}

    SCHEMA:
    {ate_schema}

    GUIDELINES:
    - Only include aspects that are explicitly mentioned (no inferred entities).
    - Prefer concrete entities over vague words (e.g., “customer service” over “experience”).
    - Merge duplicates; do not overlap spans.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ate_prompt,
                        )

In [69]:
def ote(review_text:str, ote_schema:str, ate_out:str):
    aspects_json = ate_out
    ote_prompt = f"""
    System:
    You are an expert annotator for Opinion Term Extraction (OTE). For each given aspect, extract the shortest opinion expression (≤ 3 words) directly supporting an attitude toward that aspect. Return JSON per the schema.

    User:
    TEXT:
    {review_text}

    ASPECTS (index-aligned):
    {aspects_json}

    SCHEMA:
    {ote_schema}

    GUIDELINES:
    - Link opinions to aspects.
    - “explicit” if the opinion directly modifies the aspect; otherwise “implicit”.
    - If no opinion exists for an aspect, omit it.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ote_prompt,
                        )

In [70]:
def alsc(review_text:str, alsc_schema:str, ate_out:str, ote_out:str):
    aspects_json = ate_out
    opinions_json = ote_out
    alsc_prompt = f"""
    System:
    You are an expert for Aspect-Level Sentiment Classification (ALSC). For each aspect, assign sentiment using the text and extracted opinions. Return valid JSON.

    User:
    TEXT:
    {review_text}

    ASPECTS:
    {aspects_json}

    OPINIONS:
    {opinions_json}

    SCHEMA:
    {alsc_schema}

    GUIDELINES:
    - Use evidence_span as a short verbatim quote (≤ 4 words).
    - If multiple opinions conflict, choose the dominant one; lower confidence if ambiguous.
    - overall_sentiment is the holistic tone of the whole review.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=alsc_prompt,
                        )

In [71]:
def absa_pipeline(review_text:str, ate_schema:str, ote_schema:str, alsc_schema:str) -> Dict[str,Any]:
    response = ate(review_text, ate_schema=ate_schema)
    ate_out = response['response']
    response = ote(review_text, ote_schema, ate_out)
    ote_out = response['response']
    response = alsc(review_text, alsc_schema, ate_out, ote_out)
    alsc_out = response['response']
    
    # aspects = ate_out.get("aspects", [])
    # sentiments = {s["aspect_index"]: s for s in alsc_out.get("sentiments", [])}
    # opinions_by_aspect: Dict[int, List[Dict[str,Any]]] = {}
    # for o in ote_out.get("opinions", []):
    #     opinions_by_aspect.setdefault(o["aspect_index"], []).append(o)

    # results = []
    # for i, asp in enumerate(aspects):
    #     results.append({
    #         "aspect": asp.get("term"),
    #         "category": asp.get("category"),
    #         "span": [asp.get("start"), asp.get("end")],
    #         "opinions": opinions_by_aspect.get(i, []),
    #         "sentiment": sentiments.get(i, {}).get("label"),
    #         "evidence_span": sentiments.get(i, {}).get("evidence_span"),
    #         "confidence": sentiments.get(i, {}).get("confidence"),
    #     })
    # return {
    #     "items": results,
    #     "overall_sentiment": alsc_out.get("overall_sentiment", "neutral")
    # }
    return alsc_out

In [ ]:
# review_text = df.loc[1, 'text']
# print('review_text: ', review_text)

# response = ate(review_text, ate_schema=ate_schema)
# ate_out = response['response']
# response = ote(review_text, ote_schema, ate_out)
# ote_out = response['response']
# response = alsc(review_text, alsc_schema, ate_out, ote_out)
# alsc_out = response['response']

# print(25*'*')
# print('alsc_out: ', alsc_out)

review_text:  Very unfriendly staff and service !
*************************
alsc_out:  system:
  sentiment_analysis_schema: {
    sentiments: [
      {
        aspect: "friendly",
        category: null,
        label: "positive|negative|neutral",
        evidence_span: "Very unfriendly staff and service !"
      },
      {
        aspect: "staff",
        category: null,
        label: "negative|positive|neutral",
        evidence_span: "Unfriendly staff and service ."
      },
      {
        aspect: "service",
        category: "service",
        label: "negative|positive|neutral",
        evidence_span: "Service unfriendly."
      }
    ],
    overall_sentiment: "positive|negative|neutral"
  },
  user_input:
    text: "Very unfriendly staff and service !",
    aspects: [
      {
        term: "friendly",
        category: null
      },
      {
        term: "staff",
        category: null
      },
      {
        term: "service",
        category: "service"
      }
    ]
  },
  opi

In [ ]:
review_text = df.loc[1, 'text']
print('review_text: ', review_text)

response = absa_pipeline(review_text, ate_schema, ote_schema, alsc_schema)
print(25*'*')
print(response)

### NLP feed
feed the result of NLP as the imput of LLM

In [30]:
def extract_absa_fields(absa: dict) -> dict:
    """
    Extracts fields from ABSA JSON output row.

    Expected keys:
      sentence, tokens, IOB, aspect, position, sentiment, probs, confidence
    """
    sample = ast.literal_eval(absa)
    aspects = sample.get("aspect", [])
    sentiments = sample.get("sentiment", [])

    return aspects, sentiments

In [32]:
def absa(review_text:str, absa_fields, alsc_schema):
    prompt = f"""
    System:
    You are an expert in Aspect-Based Sentiment Analysis (ABSA).
    You are given ABSA annotations produced by a RoBERTa model.
    Your job is to evaluate, correct, and improve these annotations — NOT just repeat them.

    Task Requirements:
    1. Validate and refine aspect terms (2 words max).
    2. Confirm or adjust sentiment: positive / negative / neutral.
    3. Correct aspect-opinion alignment if needed.
    4. Quote short evidence spans (≤ 6 words) from the text.
    5. Maintain consistent aspect indices.
    6. Only include explicit, text-grounded aspects (no hallucination).
    7. Output ONLY valid JSON per schema.

    Schema:
    {alsc_schema}

    User Input:
    TEXT:
    {review_text}

    RO_BERTA_ABSA_OUTPUT:
    {absa_fields}

    Instructions:
    - Review each aspect prediction. Correct where needed.
    - Drop incorrect aspects.
    - Add missing aspects only if explicit in the text.
    - Use short verbatim evidence from the review.
    - Output ONLY JSON. No explanations.
    """

    return generate(model=MODEL, 
                        prompt=prompt,
                        )

In [82]:
review_text = df.loc[1, 'text']
atepc_json = df.loc[1, 'atepc_json']
print('review_text: ', review_text)

aspects, sentiments = extract_absa_fields(atepc_json)
absa_fields = {'aspects': aspects, 'sentiments': sentiments}
result = absa(review_text, absa_fields, alsc_schema)

print(25*'*')
print(result['response'])

review_text:  Very unfriendly staff and service !
*************************
```json
{
  "sentiments": [
    {
      "aspect": "staff",
      "category": "staff",
      "label": "negative",
      "evidence_span": "Very unfriendly staff"
    },
    {
      "aspect": "service",
      "category": "service",
      "label": "negative",
      "evidence_span": "unfriendly service"
    }
  ],
  "overall_sentiment": "negative"
}
```


### NLP feed + Pipeline

In [48]:
def nlp_ate(review_text:str, nlp_aspects:list, ate_schema:str):
    ate_prompt = f"""
    System:
    You are an expert annotator for Aspect Term Extraction (ATE). 
    You are given aspects for ABSA produced by a RoBERTa model.
    Your job is to evaluate, correct, and improve these aspects — NOT just repeat them.

    User:
    TEXT:
    {review_text}

    RO_BERTA_ABSA_OUTPUT:
    {nlp_aspects}

    SCHEMA:
    {ate_schema}

    GUIDELINES:
    - Only include aspects that are explicitly mentioned (no inferred entities).
    - Prefer concrete entities over vague words (e.g., “customer service” over “experience”).
    - Merge duplicates; do not overlap spans.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ate_prompt,
                        )

In [51]:
def nlp_ote(review_text:str, nlp_sentiments:list, ote_schema:str, ate_out:str):
    aspects_json = ate_out
    ote_prompt = f"""
    System:
    You are an expert annotator for Opinion Term Extraction (OTE). 
    You are given sentiments for ABSA produced by a RoBERTa model.
    Your job is to evaluate, correct, and improve these sentiments considering the aspects — NOT just repeat them.

    User:
    TEXT:
    {review_text}

    ASPECTS:
    {aspects_json}

    RO_BERTA_ABSA_OUTPUT:
    {nlp_sentiments}

    SCHEMA:
    {ote_schema}

    GUIDELINES:
    - Link opinions to aspects.
    - “explicit” if the opinion directly modifies the aspect; otherwise “implicit”.
    - If no opinion exists for an aspect, omit it.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=ote_prompt,
                        )

In [52]:
def alsc(review_text:str, alsc_schema:str, ate_out:str, ote_out:str):
    aspects_json = ate_out
    opinions_json = ote_out
    alsc_prompt = f"""
    System:
    You are an expert for Aspect-Level Sentiment Classification (ALSC). For each aspect, assign sentiment using the text and extracted opinions. Return valid JSON.

    User:
    TEXT:
    {review_text}

    ASPECTS:
    {aspects_json}

    OPINIONS:
    {opinions_json}

    SCHEMA:
    {alsc_schema}

    GUIDELINES:
    - Use evidence_span as a short verbatim quote (≤ 4 words).
    - If multiple opinions conflict, choose the dominant one; lower confidence if ambiguous.
    - overall_sentiment is the holistic tone of the whole review.
    - Output ONLY the JSON and nothing else.
    """
    return generate(model=MODEL, 
                        prompt=alsc_prompt,
                        )

In [ ]:
def nlp_absa_pipeline(review_text:str, absa_fields:str, ate_schema:str, ote_schema:str, alsc_schema:str) -> Dict[str,Any]:
    nlp_aspects = absa_fields['aspects']
    nlp_sentiments = absa_fields['sentiments']
    response = nlp_ate(review_text, nlp_aspects, ate_schema)
    ate_out = response['response']
    response = nlp_ote(review_text, nlp_sentiments, ote_schema, ate_out)
    ote_out = response['response']
    response = alsc(review_text, alsc_schema, ate_out, ote_out)
    alsc_out = response['response']

    return alsc_out

In [81]:
review_text = df.loc[1, 'text']
atepc_json = df.loc[1, 'atepc_json']
print('review_text: ', review_text)

aspects, sentiments = extract_absa_fields(atepc_json)
absa_fields = {'aspects': aspects, 'sentiments': sentiments}
result = nlp_absa_pipeline(review_text, absa_fields, ate_schema, ote_schema, alsc_schema)

print(25*'*')
print(result)

review_text:  Very unfriendly staff and service !
*************************
```json
{
  "sentiments": [
    {
      "aspect": "staff",
      "category": "service",
      "label": "negative",
      "evidence_span": "very unfriendly staff"
    },
    {
      "aspect": "service",
      "category": "service",
      "label": "negative",
      "evidence_span": "unfriendly service"
    }
  ],
  "overall_sentiment": "negative"
}
```


### Single prompt
Try to do the absa using a single prompt

In [58]:
def single_prompt_absa(review_text:str):
    schema = """{
    "overall_sentiment": "positive | negative | neutral",
    "aspects": [
        {
        "aspect": "string (≤2 words, concise noun or noun phrase)",
        "category": "string",
        "sentiment": "positive | negative | neutral",
        "evidence_span": "string (≤3 words, exact substring from review)",
        }
    ]
    }"""

    prompt = f"""You are an expert annotator performing Aspect-Based Sentiment Analysis (ABSA) on google reviews for places.
    Return only the JSON that conforms to the provided schema.

    schema:
    {schema}

    review text:
    {review_text}

    Rules:
    - Only consider items which the aspect and sentiment are explicitly stated in the review.
    - Identify distinct aspects mentioned in the review.
    - For each aspect, provide: aspect, optional broad category of aspect, sentiment (positive/negative/neutral), an evidence_span (verbatim snippet).
    - Set overall_sentiment from the holistic tone of the review (not just the average).
    - If the review is empty or non-linguistic, return aspects=[] and overall_sentiment="neutral".
    - Do NOT include extra fields. Keep evidence_span short and exactly quoted from the review text.
    - Try to return maximum 2 words as aspect.
    - The evidence_span are supposed to be less than 3 words.
    """
    return generate(model=MODEL, 
                        prompt=prompt,
                        )

In [80]:
review_text = df.loc[1, 'text']
print('review_text: ', review_text)
print(25*'*')

res = single_prompt_absa(review_text)
print(res['response'])

review_text:  Very unfriendly staff and service !
*************************
To perform Aspect-Based Sentiment Analysis (ABSA) on the given Google review, we need to follow a structured approach based on the specified rules:

1. **Identify Overall Sentiment**: 
   - The review "Very unfriendly staff and service !" conveys negative sentiment due to words like "unfriendly."

2. **Extract Aspects**:
   - The review mentions two aspects: "staff" and "service."
   - Both aspects are associated with a negative sentiment as indicated by the word "unfriendly."

3. **Determine Aspect Categories** (optional):
   - For "staff," a broad category could be "Staff."
   - For "service," a broad category could be "Service."

4. **Extract Evidence Span**:
   - The evidence span for both aspects is derived from the phrase "unfriendly staff and service." We can use "unfriendly staff" as an evidence span for the aspect "staff" and "unfriendly service" for "service."

5. **Compile JSON Output**:

```json
{
 

## gpt-oss:20b 

### Pipeline prompts
- aspect term extraction (ATE), which identifies the features or aspects being discussed (e.g., "food"), 
- opinion term extraction (OTE), which finds the words expressing an opinion (e.g., "delicious"), 
- aspect-level sentiment classification (ASC)

# NLP feed
feed the result of NLP as the imput of LLM